# 🛡️ IoT Device Identification : Hybrid Adversarial Training (CSV + JSON) — **SANS Borderline-SMOTE**

Ce notebook entraîne les modèles sur **DEUX datasets** (CSV + JSON) avec:
- **Dataset CSV** : IPFIX ML Instances (12 foyers, 18 classes IoT)
- **Dataset JSON** : IPFIX Records (3 mois, 17 classes IoT)
- **Curriculum Learning** (2 phases: Standard → Adversarial)
- **Early Stopping par phase**
- **Crash Test** après chaque phase (bénignes + adversaires + mélangé)
- **Macro F1-Score** et métriques par classe pour chaque phase
- **Plots de présentation** générés automatiquement à chaque phase
- **⚠️ SANS Borderline-SMOTE** — pour comparaison avec la version avec équilibrage

Tous les résultats et plots sont sauvegardés sur **Google Drive**.

# 🛠️ Phase 1 : Configuration de l'Environnement (Setup)

In [1]:
# ─── Cell 1: Setup ───────────────────────────────────────────────────────────
# Mount Google Drive FIRST so the paths in Cell 2 resolve correctly
from google.colab import drive
drive.mount('/content/drive')

# Clone the repository (pull latest if already cloned)
import os
if os.path.exists('/content/pfe'):
    !cd /content/pfe && git pull
else:
    !git clone https://github.com/yacinemkk/pfe.git /content/pfe

%cd /content/pfe

# Install dependencies
!pip install -q torch torchvision tqdm numpy pandas scikit-learn matplotlib xgboost psutil

Mounted at /content/drive
Cloning into '/content/pfe'...
remote: Enumerating objects: 597, done.
remote: Counting objects: 100% (320/320), done.
remote: Compressing objects: 100% (232/232), done.
remote: Total 597 (delta 140), reused 246 (delta 83), pack-reused 277 (from 1)
Receiving objects: 100% (597/597), 14.19 MiB | 18.44 MiB/s, done.
Resolving deltas: 100% (260/260), done.
/content/pfe


# ⚙️ Phase 2 : Configuration & Pipeline de Données

In [2]:
# ─── Cell 2: Configure ───────────────────────────────────────────────────────
# ⚠️  UPDATE THESE PATHS before pressing Run All.

# Path to your JSON IPFIX files on Google Drive.
# Expected structure:
#   JSON_DATA_DIR/
#     20-01-31(1)/ipfix_202001_fixed.json
#     20-03-31/ipfix_202003.json
#     20-04-30/ipfix_202004.json
JSON_DATA_DIR = '/content/drive/MyDrive/PFE/IPFIX_Records'

# Path to your CSV IPFIX ML Instances on Google Drive.
# Expected structure:
#   CSV_DATA_DIR/
#     home1_labeled.csv
#     home2_labeled.csv
#     ...
CSV_DATA_DIR = '/content/drive/MyDrive/PFE/IPFIX_ML_Instances'

# Where to save trained models, results.json, history.json, and plots.
# This folder is on Drive — it persists after the runtime disconnects.
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/PFE/results'

# Datasets to train on: 'csv', 'json', or 'both'
DATASETS = 'both'  # Options: 'csv', 'json', 'both'

# ─── Training hyperparameters ─────────────────────────────────────────────────
SEQ_LENGTH   = 10      # Sequence length (timesteps per sample)
EPOCHS       = 30      # Total epochs (split across 2 phases)
BATCH_SIZE   = 64      # Batch size
ADV_METHOD   = 'hybrid' # Adversarial method: none, feature, pgd, fgsm, hybrid
ADV_RATIO    = 0.4     # Ratio of adversarial samples during training
MAX_FILES    = None    # Max CSV files to load (None = all). Use small int for testing.
MAX_RECORDS  = None    # Max JSON records to load (None = all). Use small int for testing.

# ─── Nouvelles Contre-Mesures Adversariales ─────────────────────────────────────
PHASE2_EPOCHS    = 25      # Epochs Phase 2 (augmenté pour meilleure robustesse)
LAMBDA_TRADES    = 6.0     # TRADES trade-off parameter
TRADES_EPSILON   = 0.05    # Perturbation budget
TRADES_PGD_STEPS = 7       # PGD steps
USE_INPUT_TRANSFORM = True # Ajouter bruit gaussien + dropout
USE_CUTMIX       = True    # Adversarial CutMix


# ─── RAM Optimization ─────────────────────────────────────────────────────────
STRIDE = 10             # Larger stride = fewer sequences = less RAM
EVAL_SUBSAMPLE = 1000  # Max samples for adversarial eval
EVAL_BATCH_SIZE = 32   # Smaller batch for evaluation (saves RAM)

# Create results directory if needed
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print(f'CSV data:     {CSV_DATA_DIR}')
print(f'JSON data:    {JSON_DATA_DIR}')
print(f'Results dir:  {DRIVE_RESULTS_DIR}')
print(f'Datasets:     {DATASETS}')
print(f'Seq length:   {SEQ_LENGTH}')
print(f'Epochs:       {EPOCHS} (Phase 1 + Phase 2)')
print(f'Batch size:   {BATCH_SIZE}')
print(f'Adv method:   {ADV_METHOD}')
print(f'Adv ratio:    {ADV_RATIO}')

# Verify CSV data exists
import glob
csv_files = glob.glob(f'{CSV_DATA_DIR}/home*_labeled.csv')
print(f'\nFound {len(csv_files)} CSV file(s):')
for f in sorted(csv_files)[:5]:
    size_mb = os.path.getsize(f) / (1024**2)
    print(f'  {os.path.basename(f)} ({size_mb:.1f} MB)')
if len(csv_files) > 5:
    print(f'  ... and {len(csv_files) - 5} more')

# Verify JSON data exists
json_files = glob.glob(f'{JSON_DATA_DIR}/**/*.json', recursive=True)
print(f'\nFound {len(json_files)} JSON file(s):')
for f in json_files:
    size_gb = os.path.getsize(f) / (1024**3)
    print(f'  {os.path.basename(f)} ({size_gb:.1f} GB)')

CSV data:     /content/drive/MyDrive/PFE/IPFIX_ML_Instances
JSON data:    /content/drive/MyDrive/PFE/IPFIX_Records
Results dir:  /content/drive/MyDrive/PFE/results
Datasets:     both
Seq length:   10
Epochs:       30 (Phase 1 + Phase 2)
Batch size:   64
Adv method:   hybrid
Adv ratio:    0.2

Found 12 CSV file(s):
  home10_labeled.csv (1099.6 MB)
  home11_labeled.csv (215.8 MB)
  home12_labeled.csv (332.2 MB)
  home1_labeled.csv (245.0 MB)
  home2_labeled.csv (297.0 MB)
  ... and 7 more

Found 3 JSON file(s):
  ipfix_202003.json (5.7 GB)
  ipfix_202004.json (5.3 GB)
  ipfix_202001_fixed.json (6.6 GB)


In [3]:
# ─── RAM Monitoring & Cleanup Utilities ───────────────────────────────────────
import gc
import psutil
import torch
import os

def get_memory_usage():
    process = psutil.Process(os.getpid())
    ram_gb = process.memory_info().rss / (1024**3)
    gpu_gb = 0
    if torch.cuda.is_available():
        gpu_gb = torch.cuda.memory_allocated() / (1024**3)
    return ram_gb, gpu_gb

def log_memory(label=''):
    ram_gb, gpu_gb = get_memory_usage()
    print(f'  [RAM {label}] {ram_gb:.2f} GB | [GPU {label}] {gpu_gb:.2f} GB')

def aggressive_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()
    ram_gb, gpu_gb = get_memory_usage()
    print(f'  [Cleanup] RAM: {ram_gb:.2f} GB | GPU: {gpu_gb:.2f} GB')

print('RAM monitoring utilities loaded.')
log_memory('startup')

RAM monitoring utilities loaded.
  [RAM startup] 0.56 GB | [GPU startup] 0.00 GB


# 🧠 Phase 3a : Entraînement sur CSV — Un Modèle par Cellule

**Workflow optimisé pour la RAM:**
- Les datasets CSV et JSON ne sont **PAS chargés en même temps** (manque de RAM)
- Phase 3a: Charger CSV → Entraîner chaque modèle dans une cellule indépendante
- Nettoyage RAM unique entre les deux phases
- Phase 3b: Charger JSON (labelisation via adresses MAC) → Entraîner chaque modèle

**Modèles:** LSTM → BiLSTM → CNN-LSTM → XGBoost-LSTM → Transformer → CNN-BiLSTM-Transformer

Pour changer de dataset, modifiez `DATASETS` dans la Cell 2.

## 📦 Chargement du Dataset CSV
Affiche les informations détaillées du dataset CSV avant l'entraînement.


In [4]:
def load_and_display_csv_dataset(csv_data_dir, seq_length=10, stride=10, save_dir=None):
    """Charge COMPLÈTEMENT le dataset CSV, affiche les infos, et sauvegarde sur Drive."""
    import sys
    import gc
    import numpy as np
    import pickle

    sys.path.insert(0, '/content/pfe')

    print("\n" + "=" * 70)
    print("  CHARGEMENT COMPLET DU DATASET CSV")
    print("=" * 70)

    print(f"\n  Répertoire : {csv_data_dir}")
    print(f"  Seq length : {seq_length} | Stride : {stride}")

    csv_files = sorted(glob.glob(f'{csv_data_dir}/home*_labeled.csv'))
    print(f"\n  Fichiers CSV trouvés : {len(csv_files)}")

    for f in csv_files:
        size_mb = os.path.getsize(f) / (1024**2)
        print(f"    {os.path.basename(f):<30s} : {size_mb:>10.1f} MB")

    total_gb = sum(os.path.getsize(f) for f in csv_files) / (1024**3)
    print(f"\n  Taille totale : {total_gb:.2f} GB")

    # Charger le dataset complet via le vrai pipeline CSV
    print("\n  Chargement via le pipeline CSV (IoTDataProcessor)...")
    from src.data.preprocessor import IoTDataProcessor

    processor = IoTDataProcessor()
    result = processor.process_all(
        max_files=None,
        data_dir=csv_data_dir,
        seq_length=seq_length,
        stride=stride,
        apply_balancing=False,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = len(features)

    print(f"\n  {'='*70}")
    print(f"  RÉSULTAT DU CHARGEMENT")
    print(f"  {'='*70}")
    print(f"    Features ({n_continuous}) : {features[:5]}...")
    print(f"    Classes ({len(label_encoder.classes_)}) : {list(label_encoder.classes_)}")
    print(f"\n  Shapes des séquences (seq_length={seq_length}, stride={stride}) :")
    print(f"    Train : {X_train.shape}  →  {len(X_train):,} séquences")
    print(f"    Val   : {X_val.shape}  →  {len(X_val):,} séquences")
    print(f"    Test  : {X_test.shape}  →  {len(X_test):,} séquences")
    print(f"    Total : {len(X_train) + len(X_val) + len(X_test):,} séquences")

    print(f"\n  Distribution des classes (train) :")
    for cls in label_encoder.classes_:
        cls_id = label_encoder.transform([cls])[0]
        count = int(np.sum(y_train == cls_id))
        bar = '█' * max(1, count // 50)
        print(f"    {cls:<30s} : {count:>6,}  {bar}")

    # ─── Sauvegarder sur Drive ────────────────────────────────────────────
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        print(f"\n  💾 Sauvegarde du dataset CSV pré-traité sur Drive...")
        print(f"     Répertoire : {save_dir}")

        np.save(f'{save_dir}/X_train.npy', X_train)
        np.save(f'{save_dir}/X_val.npy', X_val)
        np.save(f'{save_dir}/X_test.npy', X_test)
        np.save(f'{save_dir}/y_train.npy', y_train)
        np.save(f'{save_dir}/y_val.npy', y_val)
        np.save(f'{save_dir}/y_test.npy', y_test)

        with open(f'{save_dir}/csv_metadata.pkl', 'wb') as f:
            pickle.dump({
                'features': features,
                'scaler': scaler,
                'label_encoder': label_encoder,
                'n_continuous': n_continuous,
                'seq_length': seq_length,
                'stride': stride,
            }, f)

        # Marker file to indicate preprocessing is complete
        with open(f'{save_dir}/csv_ready', 'w') as f:
            f.write('ready')

        saved_gb = (X_train.nbytes + X_val.nbytes + X_test.nbytes +
                    y_train.nbytes + y_val.nbytes + y_test.nbytes) / (1024**3)
        print(f"  ✅ Dataset CSV sauvegardé ({saved_gb:.2f} GB)")
        print(f"     Fichiers : X_train, X_val, X_test, y_train, y_val, y_test, csv_metadata.pkl")

    print(f"\n  {'='*70}")
    print(f"  ✅ Dataset CSV chargé complètement en RAM")
    print(f"  {'='*70}\n")

    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }

# ─── CSV preprocessing directory on Drive ─────────────────────────────
CSV_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/csv'

if DATASETS in ['csv', 'both']:
    csv_data = load_and_display_csv_dataset(
        CSV_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
        save_dir=CSV_PREPROCESSED_DIR
    )
else:
    csv_data = None
    print('Skipping CSV dataset loading — DATASETS is not csv or both')



  CHARGEMENT COMPLET DU DATASET CSV

  Répertoire : /content/drive/MyDrive/PFE/IPFIX_ML_Instances
  Seq length : 10 | Stride : 10

  Fichiers CSV trouvés : 12
    home10_labeled.csv             :     1099.6 MB
    home11_labeled.csv             :      215.8 MB
    home12_labeled.csv             :      332.2 MB
    home1_labeled.csv              :      245.0 MB
    home2_labeled.csv              :      297.0 MB
    home3_labeled.csv              :      354.7 MB
    home4_labeled.csv              :     1541.7 MB
    home5_labeled.csv              :      366.6 MB
    home6_labeled.csv              :      932.0 MB
    home7_labeled.csv              :      527.0 MB
    home8_labeled.csv              :      317.2 MB
    home9_labeled.csv              :      238.2 MB

  Taille totale : 6.32 GB

  Chargement via le pipeline CSV (IoTDataProcessor)...
CSV-NATIVE IPFIX ML INSTANCES PREPROCESSOR (Pipeline 4 Etapes)

[ETAPE 1] Filtrage et adaptation au SDN...
  1.1 Chargement des fichiers CSV...
 

In [5]:
# ─── Training Helper Function ────────────────────────────────────────────────
import subprocess
import sys

def train_model(MODEL, dataset_type, data_dir):
    """Train a single model on a single dataset (WITHOUT Borderline-SMOTE)."""
    print(f'\n{"="*60}')
    print(f'  Training: {MODEL.upper()} on {dataset_type.upper()} (NO SMOTE)')
    print(f'{"="*60}\n')

    # ─── GPU Check ───────────────────────────────────────────────────────
    try:
        import torch
        if torch.cuda.is_available():
            gpu_count = torch.cuda.device_count()
            gpu_name = torch.cuda.get_device_name(0)
            gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            print(f'  🚀 CUDA Available: YES')
            print(f'  📌 GPU: {gpu_name} ({gpu_mem:.1f} GB)')
            print(f'  🔢 GPU Count: {gpu_count}')
            print(f'  ⚡ CUDA Version: {torch.version.cuda}')
        else:
            print(f'  ⚠️  CUDA NOT available — training on CPU')
            print(f'  💡 Tip: In Colab, go to Runtime > Change runtime type > Hardware accelerator > GPU')
    except ImportError:
        print(f'  ⚠️  PyTorch not installed')
    print()

    model_dir = f'{DRIVE_RESULTS_DIR}/models/{MODEL}_{dataset_type}_nor'
    max_arg = f'--max_files {MAX_FILES}' if dataset_type == 'csv' and MAX_FILES else ''
    max_arg = f'--max_records {MAX_RECORDS}' if dataset_type == 'json' and MAX_RECORDS else max_arg

    # ─── Preprocessed data directory ─────────────────────────────────────
    preprocessed_dir = f'{DRIVE_RESULTS_DIR}/preprocessed/{dataset_type}'
    preprocessed_arg = f'--preprocessed_dir "{preprocessed_dir}"'

    cmd = (
        f'python train_adversarial.py '
        f'--model {MODEL} '
        f'--seq_length {SEQ_LENGTH} '
        f'--adv_method {ADV_METHOD} '
        f'--adv_ratio {ADV_RATIO} '
        f'--epochs {EPOCHS} '
        f'--batch_size {BATCH_SIZE} '
        f'--dataset {dataset_type} '
        f'--data_dir "{data_dir}" '
        f'--results_dir "{model_dir}" '
        f'--phase_checkpoints '
        f'--no_balancing '
        f'--use_class_weights '
        f'{preprocessed_arg} '
        f'{max_arg}'
    )
    print(f'  CMD: {cmd[:80]}...')
    print(f'\n  ── Training Output (live) ──────────────────────────────\n')

    # Stream output in real-time so epochs are visible as they happen
    process = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        text=True, bufsize=1, universal_newlines=True
    )

    # Read stdout line by line in real-time
    for line in process.stdout:
        print(line, end='')
        sys.stdout.flush()

    # Wait for process to finish and capture stderr
    _, stderr = process.communicate()
    returncode = process.returncode

    print(f'\n  ─────────────────────────────────────────────────────────\n')

    if returncode == 0:
        print(f'  ✅ {MODEL.upper()} SAVED to {model_dir}')
    else:
        print(f'  ❌ {MODEL.upper()} FAILED (exit code: {returncode})')
        if stderr:
            print(f'\n  STDERR:\n{stderr[-3000:]}')

def verify_drive_save(MODEL, dataset_type):
    """Verify files are saved to Drive."""
    _dir = f'{DRIVE_RESULTS_DIR}/models/{MODEL}_{dataset_type}_nor'
    _files = glob.glob(f'{_dir}/*')
    if _files:
        print(f'  ✅ Drive/{MODEL}_{dataset_type}_nor: {len(_files)} file(s) saved')
        for _f in sorted(_files)[:5]:
            print(f'      - {os.path.basename(_f)}')
    else:
        print(f'  ⚠️  Drive/{MODEL}_{dataset_type}_nor: no files found')


# 🔍 Sensibilité Adversarial — Méthode d'Entraînement Améliorée

**Amélioration clé:** Mise à jour de l'analyse de sensibilité **chaque epoch** en Phase 2.

**Avant:** Sensibilité recalculée tous les 3 epochs → attaques obsolètes
**Après:** Sensibilité recalculée chaque epoch → attaques adaptées au modèle actuel

**Comment ça marche:**
1. À chaque epoch Phase 2, on analyse les vulnérabilités du modèle actuel
2. On identifie les features les plus sensibles (celles qui causent le plus de drop d'accuracy)
3. On génère des attaques ciblées sur ces features
4. Le modèle apprend à résister à ces attaques spécifiques

**Affichage pendant l'entraînement:**
```
  [Sensitivity Update] Re-computing sensitivity analysis (n=1000)...
    → packetTotalCount (Mimic_Mean): drop=0.4523
    → octetTotalCount (Zero): drop=0.3210
    → averageInterarrivalTime (Mimic_95th): drop=0.2150
```

In [ ]:
# ─── Sensitivity Analysis Visualization ───────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import json
from pathlib import Path

def plot_sensitivity_from_phase_report(results_dir=None):
    """Plot sensitivity analysis results from phase_report.json."""
    if results_dir is None:
        results_dir = DRIVE_RESULTS_DIR
    
    results_path = Path(results_dir) / 'models'
    
    if not results_path.exists():
        print('No results found yet — run training first.')
        return
    
    for model_dir in sorted(results_path.iterdir()):
        if not model_dir.is_dir():
            continue
        
        report_file = model_dir / 'phase_report.json'
        if not report_file.exists():
            continue
        
        with open(report_file) as f:
            report = json.load(f)
        
        print(f"\n{'='*80}")
        print(f"  📁 {model_dir.name}")
        print(f"{'='*80}")
        
        for phase_key in ['phase_1', 'phase_2']:
            if phase_key not in report:
                continue
            
            phase_data = report[phase_key]
            
            print(f"\n  {phase_key.upper()}:")
            for test_name in ['test1_benign', 'test2_adversarial', 'test3_mixed']:
                if test_name in phase_data:
                    m = phase_data[test_name]
                    acc = m.get('accuracy', 0)
                    f1 = m.get('f1_score', 0)
                    rr = m.get('robustness_ratio', 1.0)
                    label = test_name.replace('test1_benign', 'Bénignes').replace('test2_adversarial', 'Adversaires').replace('test3_mixed', 'Mélangé')
                    rr_str = f'  RR={rr:.4f}' if 'adversarial' in test_name else ''
                    print(f"    {label:<15s}: Acc={acc:.4f}  F1={f1:.4f}{rr_str}")
        
        # Plot robustness comparison
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        phases = []
        acc_clean = []
        acc_adv = []
        rr_vals = []
        
        for phase_key in ['phase_1', 'phase_2']:
            if phase_key not in report:
                continue
            phases.append(phase_key.replace('_', ' ').title())
            pd_data = report[phase_key]
            acc_clean.append(pd_data.get('test1_benign', {}).get('accuracy', 0))
            acc_adv.append(pd_data.get('test2_adversarial', {}).get('accuracy', 0))
            rr_vals.append(pd_data.get('test2_adversarial', {}).get('robustness_ratio', 0))
        
        if len(phases) == 0:
            plt.close(fig)
            continue
        
        # Plot 1: Accuracy comparison
        x = range(len(phases))
        w = 0.35
        axes[0].bar([i - w/2 for i in x], acc_clean, w, label='Bénignes', color='#2ecc71', edgecolor='white')
        axes[0].bar([i + w/2 for i in x], acc_adv, w, label='Adversaires', color='#e74c3c', edgecolor='white')
        axes[0].set_xlabel('Phase', fontsize=12)
        axes[0].set_ylabel('Accuracy', fontsize=12)
        axes[0].set_title('Accuracy: Bénignes vs Adversaires', fontsize=13, fontweight='bold')
        axes[0].set_xticks(list(x))
        axes[0].set_xticklabels(phases, fontsize=10)
        axes[0].legend(fontsize=10)
        axes[0].set_ylim(0, 1.05)
        for i, (c, a) in enumerate(zip(acc_clean, acc_adv)):
            axes[0].text(i - w/2, c + 0.02, f'{c:.2%}', ha='center', fontsize=9, fontweight='bold')
            axes[0].text(i + w/2, a + 0.02, f'{a:.2%}', ha='center', fontsize=9, fontweight='bold', color='#e74c3c')
        
        # Plot 2: Robustness Ratio
        colors = ['#3498db' if rr > 0.5 else '#e67e22' for rr in rr_vals]
        axes[1].bar(x, rr_vals, color=colors, edgecolor='white', width=0.5)
        axes[1].axhline(y=0.75, color='green', linestyle='--', alpha=0.7, label='Objectif (75%)')
        axes[1].set_xlabel('Phase', fontsize=12)
        axes[1].set_ylabel('Robustness Ratio', fontsize=12)
        axes[1].set_title('Robustness Ratio par Phase', fontsize=13, fontweight='bold')
        axes[1].set_xticks(list(x))
        axes[1].set_xticklabels(phases, fontsize=10)
        axes[1].set_ylim(0, 1.05)
        axes[1].legend(fontsize=10)
        for i, rr in enumerate(rr_vals):
            axes[1].text(i, rr + 0.02, f'{rr:.2%}', ha='center', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(model_dir / 'fig_sensitivity_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  📊 Plot saved: {model_dir / 'fig_sensitivity_comparison.png'}")

print('Sensitivity visualization function loaded.')
print('Run: plot_sensitivity_from_phase_report() after training to display results.')


In [6]:
# ─── MODEL 1: LSTM on CSV ─────────────────────────────────────────────
MODEL = 'lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')



################################################################################
  PHASE 3a — LSTM on CSV
################################################################################

  [RAM before_lstm_csv] 3.48 GB | [GPU before_lstm_csv] 0.00 GB

  Training: LSTM on CSV (NO SMOTE)



AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [ ]:
# ─── MODEL 2: BiLSTM on CSV ───────────────────────────────────────────
MODEL = 'bilstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


In [ ]:
# ─── MODEL 3: CNN-LSTM on CSV ───────────────────────────────────
MODEL = 'cnn_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


In [ ]:
# ─── MODEL 4: XGBoost-LSTM on CSV ───────────────────────────────
MODEL = 'xgboost_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


## 🔍 Vérification de la Tokenization (CSV)
Étape importante avant d'entraîner les modèles **Transformer** et **CNN-BiLSTM-Transformer**.


In [ ]:
def verify_tokenization_for_transformers(seq_length=10, input_size=37):
    """Vérifie que la tokenization fonctionne avec les modèles transformer."""
    import torch
    import numpy as np

    sys.path.insert(0, '/content/pfe')

    from src.data.tokenizer import IoTTokenizer, SimpleTokenizer, create_tokenizer
    from src.models.transformer import TransformerClassifier, NLPTransformerClassifier
    from src.models.cnn_bilstm_transformer import CNNBiLSTMTransformerClassifier
    from config.config import CNN_BILSTM_TRANSFORMER_CONFIG, TRANSFORMER_CONFIG

    print("\n" + "=" * 70)
    print("  VÉRIFICATION DE LA TOKENIZATION")
    print("=" * 70)

    # 1. Créer le tokenizer
    print("\n  [1] Création du tokenizer...")
    tokenizer = create_tokenizer()
    print(f"      Tokenizer : {type(tokenizer).__name__}")

    # 2. Simuler des données d'entrée pour les tests
    print("\n  [2] Simulation des données d'entrée...")
    num_classes = 18  # CSV classes
    batch_size = 2

    # Données brutes (batch, seq_len, input_size)
    X_raw = np.random.randn(batch_size, seq_length, input_size).astype(np.float32)

    # 3. Tester TransformerClassifier (input_size-based, pas de tokenization)
    print("\n  [3] Vérification avec TransformerClassifier...")
    transformer_model = TransformerClassifier(
        input_size=input_size,
        num_classes=num_classes,
        seq_length=seq_length,
        config=TRANSFORMER_CONFIG
    )
    x_tensor = torch.FloatTensor(X_raw)
    try:
        out = transformer_model(x_tensor)
        print(f"      ✅ TransformerClassifier : input {x_tensor.shape} → output {out.shape}")
        transformer_ok = True
    except Exception as e:
        print(f"      ❌ TransformerClassifier échec : {e}")
        transformer_ok = False

    # 4. Tester NLPTransformerClassifier (token-based)
    print("\n  [4] Vérification avec NLPTransformerClassifier...")
    vocab_size = 52000
    nlp_transformer = NLPTransformerClassifier(
        vocab_size=vocab_size,
        num_classes=num_classes,
        max_seq_length=seq_length
    )
    x_tok_tensor = torch.randint(0, vocab_size, (batch_size, seq_length))
    try:
        out = nlp_transformer(x_tok_tensor)
        print(f"      ✅ NLPTransformerClassifier : input {x_tok_tensor.shape} → output {out.shape}")
        nlp_ok = True
    except Exception as e:
        print(f"      ❌ NLPTransformerClassifier échec : {e}")
        nlp_ok = False

    # 5. Tester CNNBiLSTMTransformerClassifier
    print("\n  [5] Vérification avec CNNBiLSTMTransformerClassifier...")
    hybrid_model = CNNBiLSTMTransformerClassifier(
        input_size=input_size,
        num_classes=num_classes,
        seq_length=seq_length,
        config=CNN_BILSTM_TRANSFORMER_CONFIG
    )
    try:
        out = hybrid_model(x_tensor)
        print(f"      ✅ CNNBiLSTMTransformerClassifier : input {x_tensor.shape} → output {out.shape}")
        hybrid_ok = True
    except Exception as e:
        print(f"      ❌ CNNBiLSTMTransformerClassifier échec : {e}")
        hybrid_ok = False

    # Résumé
    print("\n" + "=" * 70)
    print("  RÉSUMÉ DE LA VÉRIFICATION")
    print("=" * 70)
    print(f"    Tokenizer créé           : {'✅ OK' if tokenizer else '❌ Échec'}")
    print(f"    Vocabulaire size         : {vocab_size:,} tokens")
    print(f"    TransformerClassifier    : {'✅ Compatible (features brutes)' if transformer_ok else '❌ Incompatible'}")
    print(f"    NLPTransformerClassifier : {'✅ Compatible (tokens)' if nlp_ok else '❌ Incompatible'}")
    print(f"    CNNBiLSTMTransformer     : {'✅ Compatible (features brutes)' if hybrid_ok else '❌ Incompatible'}")
    print("=" * 70 + "\n")

    return {
        'tokenizer': tokenizer,
        'transformer_ok': transformer_ok,
        'nlp_transformer_ok': nlp_ok,
        'hybrid_ok': hybrid_ok,
        'vocab_size': vocab_size
    }

if DATASETS in ['csv', 'both']:
    tokenization_results_csv = verify_tokenization_for_transformers(seq_length=SEQ_LENGTH, input_size=37)
else:
    print('Skipping tokenization verification for CSV')


In [ ]:
# ─── MODEL 5: Transformer on CSV ────────────────────────────────
MODEL = 'transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


In [ ]:
# ─── MODEL 6: CNN-BiLSTM-Transformer on CSV ─────────────────────
MODEL = 'cnn_bilstm_transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — {MODEL.upper()} on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    train_model(MODEL, 'csv', CSV_DATA_DIR)
    verify_drive_save(MODEL, 'csv')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')

print(f'\n{"="*80}')
print('  ✅ PHASE 3a COMPLETE — ALL 6 MODELS TRAINED ON CSV')
print(f'{"="*80}')


# 🧹 Nettoyage RAM entre CSV et JSON

Exécutez cette cellule une seule fois après avoir terminé tous les modèles CSV.

In [ ]:
# ─── CLEANUP RAM BEFORE JSON PHASE ──────────────────────────────────────
print('\n' + '='*80)
print('  🧹 CLEANING RAM BEFORE JSON PHASE...')
print('='*80 + '\n')

aggressive_cleanup()

print('\n✅ RAM cleaned. Ready for JSON phase.')


# 🧠 Phase 3b : Entraînement sur JSON — Un Modèle par Cellule

**Le JSON est chargé avec labelisation par adresses MAC** (sourceMacAddress / destinationMacAddress).
Chaque cellule entraîne un modèle sur JSON indépendamment.

## 📦 Chargement du Dataset JSON
Affiche les informations détaillées du dataset JSON chargé avant l'entraînement.


In [ ]:
def load_and_display_json_dataset(json_data_dir, seq_length=10, stride=10, max_records=None, save_dir=None):
    """Charge COMPLÈTEMENT le dataset JSON, affiche les infos, et sauvegarde sur Drive."""
    import sys
    import gc
    import numpy as np
    import pickle

    sys.path.insert(0, '/content/pfe')

    print("\n" + "=" * 70)
    print("  CHARGEMENT COMPLET DU DATASET JSON")
    print("=" * 70)

    print(f"\n  Répertoire : {json_data_dir}")
    print(f"  Seq length : {seq_length} | Stride : {stride}")
    print(f"  Max records: {max_records if max_records else 'Tous'}")

    json_files = sorted(glob.glob(f'{json_data_dir}/**/*.json', recursive=True))
    print(f"\n  Fichiers JSON trouvés : {len(json_files)}")

    for f in json_files:
        size_gb = os.path.getsize(f) / (1024**3)
        print(f"    {os.path.basename(f):<30s} : {size_gb:>10.2f} GB")

    total_gb = sum(os.path.getsize(f) for f in json_files) / (1024**3)
    print(f"\n  Taille totale : {total_gb:.2f} GB")

    # Charger le dataset complet via le vrai pipeline JSON
    print("\n  Chargement via le pipeline JSON (JsonIoTDataProcessor)...")
    from src.data.json_preprocessor import JsonIoTDataProcessor

    processor = JsonIoTDataProcessor()
    result = processor.process_all(
        data_dir=json_data_dir,
        seq_length=seq_length,
        stride=stride,
        max_records=max_records,
        apply_balancing=False,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = 36  # JSON pipeline features

    print(f"\n  {'='*70}")
    print(f"  RÉSULTAT DU CHARGEMENT")
    print(f"  {'='*70}")
    print(f"    Features ({len(features)}) : {features[:5]}...")
    print(f"    Classes ({len(label_encoder.classes_)}) : {list(label_encoder.classes_)}")
    print(f"\n  Shapes des séquences (seq_length={seq_length}, stride={stride}) :")
    print(f"    Train : {X_train.shape}  →  {len(X_train):,} séquences")
    print(f"    Val   : {X_val.shape}  →  {len(X_val):,} séquences")
    print(f"    Test  : {X_test.shape}  →  {len(X_test):,} séquences")
    print(f"    Total : {len(X_train) + len(X_val) + len(X_test):,} séquences")

    print(f"\n  Distribution des classes (train) :")
    for cls in label_encoder.classes_:
        cls_id = label_encoder.transform([cls])[0]
        count = int(np.sum(y_train == cls_id))
        bar = '█' * max(1, count // 50)
        print(f"    {cls:<30s} : {count:>6,}  {bar}")

    # ─── Sauvegarder sur Drive ────────────────────────────────────────────
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        print(f"\n  💾 Sauvegarde du dataset JSON pré-traité sur Drive...")
        print(f"     Répertoire : {save_dir}")

        np.save(f'{save_dir}/X_train.npy', X_train)
        np.save(f'{save_dir}/X_val.npy', X_val)
        np.save(f'{save_dir}/X_test.npy', X_test)
        np.save(f'{save_dir}/y_train.npy', y_train)
        np.save(f'{save_dir}/y_val.npy', y_val)
        np.save(f'{save_dir}/y_test.npy', y_test)

        with open(f'{save_dir}/json_metadata.pkl', 'wb') as f:
            pickle.dump({
                'features': features,
                'scaler': scaler,
                'label_encoder': label_encoder,
                'n_continuous': n_continuous,
                'seq_length': seq_length,
                'stride': stride,
            }, f)

        # Marker file to indicate preprocessing is complete
        with open(f'{save_dir}/json_ready', 'w') as f:
            f.write('ready')

        saved_gb = (X_train.nbytes + X_val.nbytes + X_test.nbytes +
                    y_train.nbytes + y_val.nbytes + y_test.nbytes) / (1024**3)
        print(f"  ✅ Dataset JSON sauvegardé ({saved_gb:.2f} GB)")
        print(f"     Fichiers : X_train, X_val, X_test, y_train, y_val, y_test, json_metadata.pkl")

    print(f"\n  {'='*70}")
    print(f"  ✅ Dataset JSON chargé complètement en RAM")
    print(f"  {'='*70}\n")

    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }

# ─── JSON preprocessing directory on Drive ─────────────────────────────
JSON_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/json'

if DATASETS in ['json', 'both']:
    json_data = load_and_display_json_dataset(
        JSON_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
        max_records=MAX_RECORDS, save_dir=JSON_PREPROCESSED_DIR
    )
else:
    json_data = None
    print('Skipping JSON dataset loading — DATASETS is not json or both')


In [ ]:
# ─── MODEL 1: LSTM on JSON ─────────────────────────────────────────────
MODEL = 'lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL 2: BiLSTM on JSON ───────────────────────────────────────────
MODEL = 'bilstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL 3: CNN-LSTM on JSON ───────────────────────────────────
MODEL = 'cnn_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL 4: XGBoost-LSTM on JSON ───────────────────────────────
MODEL = 'xgboost_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


## 🔍 Vérification de la Tokenization (JSON)
Étape importante avant d'entraîner les modèles **Transformer** et **CNN-BiLSTM-Transformer**.


In [ ]:
def verify_tokenization_for_transformers_json(seq_length=10, input_size=36):
    """Vérifie que la tokenization fonctionne avec les modèles transformer (JSON pipeline)."""
    import torch
    import numpy as np

    sys.path.insert(0, '/content/pfe')

    from src.data.tokenizer import IoTTokenizer, SimpleTokenizer, create_tokenizer
    from src.models.transformer import TransformerClassifier, NLPTransformerClassifier
    from src.models.cnn_bilstm_transformer import CNNBiLSTMTransformerClassifier
    from config.config import CNN_BILSTM_TRANSFORMER_CONFIG, TRANSFORMER_CONFIG

    print("\n" + "=" * 70)
    print("  VÉRIFICATION DE LA TOKENIZATION (JSON)")
    print("=" * 70)

    # 1. Créer le tokenizer
    print("\n  [1] Création du tokenizer...")
    tokenizer = create_tokenizer()
    print(f"      Tokenizer : {type(tokenizer).__name__}")

    # 2. Simuler des données d'entrée pour les tests
    print("\n  [2] Simulation des données d'entrée...")
    num_classes = 17  # JSON classes
    batch_size = 2

    # Données brutes (batch, seq_len, input_size)
    X_raw = np.random.randn(batch_size, seq_length, input_size).astype(np.float32)

    # 3. Tester TransformerClassifier (input_size-based, pas de tokenization)
    print("\n  [3] Vérification avec TransformerClassifier...")
    transformer_model = TransformerClassifier(
        input_size=input_size,
        num_classes=num_classes,
        seq_length=seq_length,
        config=TRANSFORMER_CONFIG
    )
    x_tensor = torch.FloatTensor(X_raw)
    try:
        out = transformer_model(x_tensor)
        print(f"      ✅ TransformerClassifier : input {x_tensor.shape} → output {out.shape}")
        transformer_ok = True
    except Exception as e:
        print(f"      ❌ TransformerClassifier échec : {e}")
        transformer_ok = False

    # 4. Tester NLPTransformerClassifier (token-based)
    print("\n  [4] Vérification avec NLPTransformerClassifier...")
    vocab_size = 52000
    nlp_transformer = NLPTransformerClassifier(
        vocab_size=vocab_size,
        num_classes=num_classes,
        max_seq_length=seq_length
    )
    x_tok_tensor = torch.randint(0, vocab_size, (batch_size, seq_length))
    try:
        out = nlp_transformer(x_tok_tensor)
        print(f"      ✅ NLPTransformerClassifier : input {x_tok_tensor.shape} → output {out.shape}")
        nlp_ok = True
    except Exception as e:
        print(f"      ❌ NLPTransformerClassifier échec : {e}")
        nlp_ok = False

    # 5. Tester CNNBiLSTMTransformerClassifier
    print("\n  [5] Vérification avec CNNBiLSTMTransformerClassifier...")
    hybrid_model = CNNBiLSTMTransformerClassifier(
        input_size=input_size,
        num_classes=num_classes,
        seq_length=seq_length,
        config=CNN_BILSTM_TRANSFORMER_CONFIG
    )
    try:
        out = hybrid_model(x_tensor)
        print(f"      ✅ CNNBiLSTMTransformerClassifier : input {x_tensor.shape} → output {out.shape}")
        hybrid_ok = True
    except Exception as e:
        print(f"      ❌ CNNBiLSTMTransformerClassifier échec : {e}")
        hybrid_ok = False

    # Résumé
    print("\n" + "=" * 70)
    print("  RÉSUMÉ DE LA VÉRIFICATION")
    print("=" * 70)
    print(f"    Tokenizer créé           : {'✅ OK' if tokenizer else '❌ Échec'}")
    print(f"    Vocabulaire size         : {vocab_size:,} tokens")
    print(f"    TransformerClassifier    : {'✅ Compatible (features brutes)' if transformer_ok else '❌ Incompatible'}")
    print(f"    NLPTransformerClassifier : {'✅ Compatible (tokens)' if nlp_ok else '❌ Incompatible'}")
    print(f"    CNNBiLSTMTransformer     : {'✅ Compatible (features brutes)' if hybrid_ok else '❌ Incompatible'}")
    print("=" * 70 + "\n")

    return {
        'tokenizer': tokenizer,
        'transformer_ok': transformer_ok,
        'nlp_transformer_ok': nlp_ok,
        'hybrid_ok': hybrid_ok,
        'vocab_size': vocab_size
    }

if DATASETS in ['json', 'both']:
    tokenization_results_json = verify_tokenization_for_transformers_json(seq_length=SEQ_LENGTH, input_size=36)
else:
    print('Skipping tokenization verification for JSON')


In [ ]:
# ─── MODEL 5: Transformer on JSON ────────────────────────────────
MODEL = 'transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL 6: CNN-BiLSTM-Transformer on JSON ─────────────────────
MODEL = 'cnn_bilstm_transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — {MODEL.upper()} on JSON (MAC address labeling)')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    train_model(MODEL, 'json', JSON_DATA_DIR)
    verify_drive_save(MODEL, 'json')
else:
    print('Skipping JSON dataset')

aggressive_cleanup()

print(f'\n✅ {MODEL.upper()} on JSON DONE')

print(f'\n{"="*80}')
print('  ✅ ALL 6 MODELS TRAINED ON BOTH DATASETS — COMPLETE!')
print(f'{"="*80}')


# 📊 Phase 4 : Visualisation de TOUS les Résultats

Affiche les plots générés pendant l'entraînement — directement depuis Drive.

Les fichiers suivants sont générés pour chaque modèle, chaque dataset et chaque phase :
- `fig_device_identification_scores_phaseN.png` — P/R/F1 par appareil
- `fig_adversarial_effect_phaseN.png` — Impact des attaques par appareil
- `fig_robustness_summary_phaseN.png` — Accuracy + Macro-F1 clean vs attaques
- `fig_training_history.png` — Courbes loss/accuracy

In [ ]:
# ─── Cell: Display ALL Plots ─────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from IPython.display import display, HTML

results_dir = Path(DRIVE_RESULTS_DIR) / 'models'
if not results_dir.exists():
    print('No results found yet — run the training cells first.')
else:
    result_dirs = sorted(results_dir.iterdir())

    for rd in result_dirs:
        if not rd.is_dir():
            continue

        model_name = rd.name
        print(f"\n{'='*80}")
        print(f"  📁 {model_name}")
        print(f"{'='*80}")

        # Show results summary
        results_file = rd / 'results.json'
        if results_file.exists():
            with open(results_file) as f:
                res = json.load(f)
            print(f"  Model: {res.get('model_type', '?')} | Input: {res.get('input_size', '?')} | Classes: {res.get('num_classes', '?')}")
            print(f"  Clean Accuracy: {res.get('test_accuracy_clean', 0):.4f}")
            if 'clean_metrics' in res:
                cm = res['clean_metrics']
                print(f"  Macro F1: {cm.get('macro_f1', 0):.4f} | Macro P: {cm.get('macro_precision', 0):.4f} | Macro R: {cm.get('macro_recall', 0):.4f}")
            if 'robustness_ratios' in res:
                print(f"  Robustness Ratios: {res['robustness_ratios']}")

        # Display all plots
        plot_files = sorted(rd.glob('fig_*.png'))
        if plot_files:
            for pf in plot_files:
                print(f"\n  📊 {pf.name}")
                fig, ax = plt.subplots(figsize=(16, 8))
                img = mpimg.imread(str(pf))
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(pf.stem.replace('_', ' ').title(), fontsize=14, fontweight='bold')
                plt.tight_layout()
                plt.show()
        else:
            print('  No plots found.')

        # Show curriculum report if available
        report_file = rd / 'curriculum_report.json'
        if report_file.exists():
            with open(report_file) as f:
                report = json.load(f)
            print(f"\n  📋 Curriculum Report:")
            for phase_key, phase_data in report.get('phases', {}).items():
                phase_num = phase_data.get('phase', '?')
                clean = phase_data.get('clean', {})
                feat = phase_data.get('feature_attack', {})
                pgd = phase_data.get('sequence_pgd', {})
                fgsm = phase_data.get('sequence_fgsm', {})
                print(f"    Phase {phase_num}:")
                print(f"      Clean  — Acc: {clean.get('accuracy', 0):.4f}  F1: {clean.get('f1_score', 0):.4f}  Macro-F1: {clean.get('macro_f1', 0):.4f}")
                if feat:
                    print(f"      FeatAdv — Acc: {feat.get('accuracy', 0):.4f}  F1: {feat.get('f1_score', 0):.4f}  RR: {feat.get('robustness_ratio', 0):.4f}")
                if pgd:
                    print(f"      SeqPGD — Acc: {pgd.get('accuracy', 0):.4f}  F1: {pgd.get('f1_score', 0):.4f}  RR: {pgd.get('robustness_ratio', 0):.4f}")
                if fgsm:
                    print(f"      SeqFGSM — Acc: {fgsm.get('accuracy', 0):.4f}  F1: {fgsm.get('f1_score', 0):.4f}  RR: {fgsm.get('robustness_ratio', 0):.4f}")

print('\n\n✅ All results displayed.')

# 🛡️ Phase 4b : Entraînement avec TRADES (Alternative à l'Adversarial Training classique)

TRADES (TRadeoff-inspired Adversarial DEfenseS) est une méthode d'entraînement adversarial qui:
- Optimise la loss standard sur les exemples clean
- Maximise la cohérence des prédictions entre exemples clean et adversariaux
- Contrôle le trade-off avec un paramètre lambda

**Formule:** Loss = CE(x, y) + λ × KL(f(x) || f(x_adv))

**Avantages vs votre méthode actuelle:**
- Pas besoin de générer les attaques avant l'entraînement
- Les exemples adversariaux sont générés dynamiquement pendant l'entraînement
- Meilleur contrôle du trade-off accuracy/robustesse

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRADES ADVERSARIAL TRAINING - Alternative Method for Better Robustness
# ═══════════════════════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, '/content/pfe')

import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, Dataset
import gc

# Import TRADES module
from src.adversarial.trades import TRADESTrainer, create_trades_projection_fn
from src.adversarial.input_transform import InputTransform, create_input_transform
from src.adversarial.cutmix import AdversarialCutMix, create_adversarial_cutmix
from src.models.lstm import LSTMClassifier
from src.training.trainer import IoTSequenceDataset
from config.config import LSTM_CONFIG, LEARNING_RATE, WEIGHT_DECAY

def train_with_trades(
    model_type='lstm',
    data_dict=None,
    lambda_trades=6.0,
    epsilon=0.05,
    epochs=30,
    batch_size=64,
    lr=1e-3,
    save_dir=None,
    use_class_weights=True,
):
    """
    Train a model using TRADES adversarial training.
    
    Args:
        model_type: 'lstm', 'bilstm', 'cnn_lstm', etc.
        data_dict: dict with X_train, X_val, X_test, y_train, y_val, y_test, features, etc.
        lambda_trades: TRADES trade-off parameter (typical: 1-10, higher = more robust)
        epsilon: perturbation budget for adversarial examples
        epochs: number of training epochs
        batch_size: batch size
        lr: learning rate
        save_dir: directory to save model
        use_class_weights: whether to use class weights for imbalanced data
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*70}")
    print(f"  TRAINES ADVERSARIAL TRAINING")
    print(f"{'='*70}")
    print(f"  Model: {model_type.upper()}")
    print(f"  Device: {device}")
    print(f"  Lambda TRADES: {lambda_trades}")
    print(f"  Epsilon: {epsilon}")
    print(f"  Epochs: {epochs}")
    print(f"  Batch size: {batch_size}")
    print(f"  Class weights: {use_class_weights}")
    print(f"{'='*70}\n")
    
    # Extract data
    X_train = data_dict['X_train']
    X_val = data_dict['X_val']
    X_test = data_dict['X_test']
    y_train = data_dict['y_train']
    y_val = data_dict['y_val']
    y_test = data_dict['y_test']
    features = data_dict['features']
    n_continuous = data_dict.get('n_continuous', len(features))
    
    # Get dimensions
    input_size = X_train.shape[2]
    num_classes = len(np.unique(y_train))
    
    print(f"  Input size: {input_size}")
    print(f"  Num classes: {num_classes}")
    print(f"  Train samples: {len(X_train):,}")
    print(f"  Val samples: {len(X_val):,}")
    print(f"  Test samples: {len(X_test):,}")
    
    # Create datasets
    train_dataset = IoTSequenceDataset(X_train, y_train)
    val_dataset = IoTSequenceDataset(X_val, y_val)
    test_dataset = IoTSequenceDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    # Create model
    if model_type == 'lstm':
        model = LSTMClassifier(input_size, num_classes, LSTM_CONFIG)
    elif model_type == 'bilstm':
        bilstm_config = dict(LSTM_CONFIG)
        bilstm_config['bidirectional'] = True
        model = LSTMClassifier(input_size, num_classes, bilstm_config)
    else:
        raise ValueError(f"Model type {model_type} not yet supported for TRADES")
    
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model parameters: {n_params:,}")
    
    # Create TRADES trainer
    trainer = TRADESTrainer(
        model=model,
        device=device,
        lambda_trades=lambda_trades,
        epsilon=epsilon,
        alpha=epsilon / 4,  # Step size
        num_steps=7,  # PGD steps
        verbose=True,
    )
    
    # Train with TRADES
    history = trainer.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        epochs=epochs,
        lr=lr,
        weight_decay=WEIGHT_DECAY,
        patience=7,
        save_path=save_dir,
        use_class_weights=use_class_weights,
    )
    
    # Final robustness evaluation
    print(f"\n{'='*70}")
    print(f"  FINAL ROBUSTNESS EVALUATION")
    print(f"{'='*70}\n")
    
    criterion = nn.CrossEntropyLoss()
    final_results = trainer.test_robustness(test_loader, criterion)
    
    # Save results
    if save_dir:
        import os
        import json
        os.makedirs(save_dir, exist_ok=True)
        
        results = {
            'model_type': model_type,
            'lambda_trades': lambda_trades,
            'epsilon': epsilon,
            'epochs': epochs,
            'final_results': final_results,
            'history': {k: [float(v) for v in vals] for k, vals in history.items()},
        }
        
        with open(f'{save_dir}/trades_results.json', 'w') as f:
            json.dump(results, f, indent=2)
        
        print(f"\n  💾 Results saved to: {save_dir}")
    
    # Cleanup
    del train_loader, val_loader, test_loader
    del train_dataset, val_dataset, test_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return history, final_results

print("✅ TRADES training function loaded.")
print("\nUsage:")
print("  history, results = train_with_trades(")
print("      model_type='lstm',")
print("      data_dict=csv_data,")
print("      lambda_trades=6.0,")
print("      epsilon=0.05,")
print("      epochs=30,")
print("      batch_size=64,")
print("      save_dir=f'{DRIVE_RESULTS_DIR}/models/lstm_trades_csv'")
print("  )")

In [ ]:
# ─── Exemple 1: Entraîner LSTM avec TRADES sur CSV ──────────────────────────────
# Paramètres recommandés pour améliorer la robustesse:
# - lambda_trades: 6.0 (contrôle le trade-off accuracy/robustesse)
# - epsilon: 0.05 (budget de perturbation)
# - epochs: 30-50 (plus d'epochs pour converger)

if DATASETS in ['csv', 'both'] and csv_data is not None:
    print("\n" + "#"*80)
    print("  TRAINES TRAINING - LSTM on CSV")
    print("#"*80 + "\n")
    
    log_memory('before_trades_lstm_csv')
    
    trades_history, trades_results = train_with_trades(
        model_type='lstm',
        data_dict=csv_data,
        lambda_trades=6.0,      # Augmenter pour plus de robustesse (1-10)
        epsilon=0.05,           # Budget de perturbation
        epochs=30,              # Plus d'epochs = meilleure convergence
        batch_size=64,
        lr=1e-3,
        save_dir=f'{DRIVE_RESULTS_DIR}/models/lstm_trades_csv',
        use_class_weights=True,
    )
    
    print(f"\n✅ TRADES LSTM training complete!")
    print(f"  Clean Accuracy: {trades_results['clean']['accuracy']:.4f}")
    print(f"  TRADES Robust Accuracy: {trades_results['trades_adversarial']['accuracy']:.4f}")
    print(f"  Robustness Ratio: {trades_results['trades_adversarial']['robustness_ratio']:.4f}")
    
    log_memory('after_trades_lstm_csv')
    aggressive_cleanup()
else:
    print("Skipping TRADES training - CSV data not loaded")

In [ ]:
# ─── Exemple 2: Entraîner BiLSTM avec TRADES (plus robuste) ─────────────────────
# BiLSTM est souvent plus robuste que LSTM car il capture le contexte bidirectionnel

if DATASETS in ['csv', 'both'] and csv_data is not None:
    print("\n" + "#"*80)
    print("  TRAINES TRAINING - BiLSTM on CSV")
    print("#"*80 + "\n")
    
    log_memory('before_trades_bilstm_csv')
    
    # Lambda plus élevé pour BiLSTM car il converge mieux
    trades_history, trades_results = train_with_trades(
        model_type='bilstm',
        data_dict=csv_data,
        lambda_trades=8.0,      # Plus élevé pour BiLSTM
        epsilon=0.05,
        epochs=40,              # Plus d'epochs pour BiLSTM
        batch_size=64,
        lr=1e-3,
        save_dir=f'{DRIVE_RESULTS_DIR}/models/bilstm_trades_csv',
        use_class_weights=True,
    )
    
    print(f"\n✅ TRADES BiLSTM training complete!")
    print(f"  Clean Accuracy: {trades_results['clean']['accuracy']:.4f}")
    print(f"  TRADES Robust Accuracy: {trades_results['trades_adversarial']['accuracy']:.4f}")
    print(f"  Robustness Ratio: {trades_results['trades_adversarial']['robustness_ratio']:.4f}")
    
    log_memory('after_trades_bilstm_csv')
    aggressive_cleanup()
else:
    print("Skipping TRADES training - CSV data not loaded")

# 🛡️ Phase 4c : Nouvelles Contre-Mesures Adversariales

Cette section implémente les nouvelles contre-mesures pour améliorer la robustesse:

## Contre-Mesures Implémentées

| Contre-Mesure | Description | Paramètres |
|---------------|-------------|------------|
| **TRADES** | Loss = CE + λ×KL(f(x)||f(x_adv)) | λ=6.0, ε=0.05 |
| **Input Transform** | Bruit gaussien + dropout aléatoire | σ=0.02, p=0.1 |
| **Adversarial CutMix** | Mélange clean/adversarial | α=1.0, prob=0.5 |
| **Sensitivity Update** | Mise à jour chaque epoch | - |

## Architecture d'Entraînement Phase 2

```
Batch → Input Transform → TRADES PGD Attack → Adversarial CutMix → Loss
```

## Paramètres Mis à Jour

- `ADV_RATIO`: 0.2 → **0.4** (40% d'exemples adverses)
- `PHASE2_EPOCHS`: 10 → **25** (plus d'entraînement)
- `LAMBDA_TRADES`: **6.0** (trade-off accuracy/robustesse)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ENTRAÎNEMENT AVEC NOUVELLES CONTRE-MESURES
# ═══════════════════════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, '/content/pfe')

import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
import gc
from datetime import datetime

# Imports des modules adversariaux
from src.adversarial.trades import TRADESTrainer, TRADESAttack
from src.adversarial.input_transform import create_input_transform
from src.adversarial.cutmix import create_adversarial_cutmix
from src.models.lstm import LSTMClassifier
from src.training.trainer import IoTSequenceDataset
from config.config import LSTM_CONFIG, LEARNING_RATE, WEIGHT_DECAY

def train_with_countermeasures(
    model_type='lstm',
    data_dict=None,
    lambda_trades=6.0,
    epsilon=0.05,
    alpha=0.01,
    pgd_steps=7,
    epochs_phase1=20,
    epochs_phase2=25,
    batch_size=64,
    lr=1e-3,
    use_input_transform=True,
    use_cutmix=True,
    save_dir=None,
    use_class_weights=True,
):
    """
    Entraînement avec toutes les contre-mesures activées.
    
    Phase 1: Entraînement standard (données bénignes)
    Phase 2: Entraînement adversarial avec TRADES + Input Transform + CutMix
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*80}")
    print(f"  ENTRAÎNEMENT AVEC CONTRE-MESURES ADVERSARIALES")
    print(f"{'='*80}")
    print(f"  Model: {model_type.upper()}")
    print(f"  Device: {device}")
    print(f"  Phase 1 epochs: {epochs_phase1}")
    print(f"  Phase 2 epochs: {epochs_phase2}")
    print(f"  λ_TRADES: {lambda_trades}")
    print(f"  ε (epsilon): {epsilon}")
    print(f"  Input Transform: {use_input_transform}")
    print(f"  Adversarial CutMix: {use_cutmix}")
    print(f"{'='*80}\n")
    
    # Extract data
    X_train = data_dict['X_train']
    X_val = data_dict['X_val']
    X_test = data_dict['X_test']
    y_train = data_dict['y_train']
    y_val = data_dict['y_val']
    y_test = data_dict['y_test']
    features = data_dict['features']
    n_continuous = data_dict.get('n_continuous', len(features))
    
    # Get dimensions
    input_size = X_train.shape[2]
    num_classes = len(np.unique(y_train))
    
    print(f"  Input size: {input_size}")
    print(f"  Num classes: {num_classes}")
    print(f"  Train samples: {len(X_train):,}")
    print(f"  Val samples: {len(X_val):,}")
    print(f"  Test samples: {len(X_test):,}")
    
    # Create datasets
    train_dataset = IoTSequenceDataset(X_train, y_train)
    val_dataset = IoTSequenceDataset(X_val, y_val)
    test_dataset = IoTSequenceDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    # Create model
    if model_type == 'lstm':
        model = LSTMClassifier(input_size, num_classes, LSTM_CONFIG)
    elif model_type == 'bilstm':
        bilstm_config = dict(LSTM_CONFIG)
        bilstm_config['bidirectional'] = True
        model = LSTMClassifier(input_size, num_classes, bilstm_config)
    else:
        raise ValueError(f"Model type {model_type} not supported")
    
    model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model parameters: {n_params:,}")
    
    # ─────────────────────────────────────────────────────────────────────────────
    # PHASE 1: Entraînement Standard
    # ─────────────────────────────────────────────────────────────────────────────
    print(f"\n{'─'*80}")
    print(f"  PHASE 1: ENTRAÎNEMENT STANDARD")
    print(f"{'─'*80}\n")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(epochs_phase1):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                val_loss += criterion(outputs, y_batch).item()
        
        val_loss /= len(val_loader)
        train_acc = correct / total
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print(f"  Early stopping Phase 1 at epoch {epoch+1}")
                break
        
        if (epoch + 1) % 5 == 0:
            print(f"  P1 Epoch {epoch+1}/{epochs_phase1} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}")
    
    model.load_state_dict(best_state)
    print(f"  ✓ Phase 1 complete. Best val loss: {best_val_loss:.4f}")
    
    # ─────────────────────────────────────────────────────────────────────────────
    # PHASE 2: Entraînement Adversarial avec Contre-Mesures
    # ─────────────────────────────────────────────────────────────────────────────
    print(f"\n{'─'*80}")
    print(f"  PHASE 2: ENTRAÎNEMENT ADVERSARIAL (TRADES + Transform + CutMix)")
    print(f"{'─'*80}\n")
    
    # Initialize countermeasures
    trades_attack = TRADESAttack(epsilon=epsilon, alpha=alpha, num_steps=pgd_steps)
    
    input_transform = None
    if use_input_transform:
        input_transform = create_input_transform(noise_std=0.02, dropout_prob=0.1, n_continuous_features=n_continuous)
        print(f"  Input Transform: noise_std=0.02, dropout=0.1")
    
    adv_cutmix = None
    if use_cutmix:
        adv_cutmix = create_adversarial_cutmix(alpha=1.0, prob=0.5)
        print(f"  Adversarial CutMix: α=1.0, prob=0.5")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr/2, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    import torch.nn.functional as F
    
    for epoch in range(epochs_phase2):
        model.train()
        train_loss = 0
        ce_loss_total = 0
        kl_loss_total = 0
        correct = 0
        total = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            # Apply input transform
            if input_transform:
                X_batch = input_transform(X_batch, training=True)
            
            # Generate adversarial examples with TRADES
            with torch.no_grad():
                X_adv = trades_attack.generate(model, X_batch, y_batch, device)
            
            # Apply adversarial cutmix
            if adv_cutmix and np.random.rand() < 0.5:
                X_adv, y_batch = adv_cutmix(X_batch, X_adv, y_batch)
            
            # Forward pass
            optimizer.zero_grad()
            clean_logits = model(X_batch)
            ce_loss = criterion(clean_logits, y_batch)
            
            adv_logits = model(X_adv)
            
            # TRADES KL loss
            clean_probs = F.softmax(clean_logits, dim=1)
            adv_log_probs = F.log_softmax(adv_logits, dim=1)
            kl_loss = F.kl_div(adv_log_probs, clean_probs, reduction='batchmean')
            
            loss = ce_loss + lambda_trades * kl_loss
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            ce_loss_total += ce_loss.item()
            kl_loss_total += kl_loss.item()
            _, predicted = clean_logits.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()
            
            del X_adv
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                val_loss += criterion(outputs, y_batch).item()
        
        val_loss /= len(val_loader)
        train_acc = correct / total
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print(f"  Early stopping Phase 2 at epoch {epoch+1}")
                break
        
        if (epoch + 1) % 5 == 0:
            avg_ce = ce_loss_total / len(train_loader)
            avg_kl = kl_loss_total / len(train_loader)
            print(f"  P2 Epoch {epoch+1}/{epochs_phase2} | Acc: {train_acc:.4f} | CE: {avg_ce:.4f} | KL: {avg_kl:.4f}")
    
    model.load_state_dict(best_state)
    print(f"  ✓ Phase 2 complete. Best val loss: {best_val_loss:.4f}")
    
    # ─────────────────────────────────────────────────────────────────────────────
    # Final Evaluation
    # ─────────────────────────────────────────────────────────────────────────────
    print(f"\n{'='*80}")
    print(f"  ÉVALUATION FINALE")
    print(f"{'='*80}\n")
    
    model.eval()
    correct_clean = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct_clean += predicted.eq(y_batch).sum().item()
    clean_acc = correct_clean / total
    print(f"  Clean Accuracy: {clean_acc:.4f}")
    
    # Adversarial evaluation
    correct_adv = 0
    total = 0
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        with torch.no_grad():
            X_adv = trades_attack.generate(model, X_batch, y_batch, device)
        outputs = model(X_adv)
        _, predicted = outputs.max(1)
        total += y_batch.size(0)
        correct_adv += predicted.eq(y_batch).sum().item()
    adv_acc = correct_adv / total
    robustness_ratio = adv_acc / max(clean_acc, 1e-8)
    print(f"  Adversarial Accuracy: {adv_acc:.4f}")
    print(f"  Robustness Ratio: {robustness_ratio:.4f}")
    
    results = {
        'clean_accuracy': clean_acc,
        'adversarial_accuracy': adv_acc,
        'robustness_ratio': robustness_ratio,
    }
    
    # Save
    if save_dir:
        import os
        os.makedirs(save_dir, exist_ok=True)
        torch.save(model.state_dict(), f'{save_dir}/model.pt')
        import json
        with open(f'{save_dir}/countermeasures_results.json', 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\n  💾 Model and results saved to: {save_dir}")
    
    # Cleanup
    del train_loader, val_loader, test_loader
    del train_dataset, val_dataset, test_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return results

print("✅ Fonction train_with_countermeasures chargée.")
print("\nUsage:")
print("  results = train_with_countermeasures(")
print("      model_type='lstm',")
print("      data_dict=csv_data,")
print("      lambda_trades=6.0,")
print("      epochs_phase2=25,")
print("      save_dir=f'{DRIVE_RESULTS_DIR}/models/lstm_countermeasures'")
print("  )")

## 📊 Comparaison TRADES vs Adversarial Training Classique

| Métrique | Adversarial Training Classique | TRADES |
|----------|-------------------------------|--------|
| **Génération d'attaques** | Avant l'entraînement (Phase 1) | Pendant l'entraînement (dynamique) |
| **Contrôle du trade-off** | Via `adv_ratio` (0.2) | Via `lambda` (6-10) |
| **Robustesse attendue** | ~5% (d'après vos résultats) | ~30-50% (littérature) |
| **Accuracy clean** | ~95% | ~85-90% (trade-off) |

### Conseils pour améliorer la robustesse avec votre méthode actuelle:

1. **Augmenter `adv_ratio`**: de 0.2 à 0.5-0.7
2. **Augmenter les epochs Phase 2**: de 10 à 30-50
3. **Modifier `hybrid_split`**: réduire clean de 0.6 à 0.3
4. **Utiliser des attaques plus fortes**: PGD avec plus d'itérations

### Pour utiliser TRADES à la place:

```python
# Au lieu de train_model(MODEL, 'csv', CSV_DATA_DIR)
# Utiliser:
history, results = train_with_trades(
    model_type='lstm',
    data_dict=csv_data,
    lambda_trades=6.0,
    epsilon=0.05,
    epochs=40,
    save_dir=f'{DRIVE_RESULTS_DIR}/models/lstm_trades_csv'
)
```

# 📋 Phase 5 : Tableau Récapitulatif Multi-Modèles (CSV + JSON)

Compare les performances de tous les modèles entraînés sur les deux datasets.

In [ ]:
# ─── Cell: Comparative Summary Table ─────────────────────────────────────────
import json
from pathlib import Path

results_dir = Path(DRIVE_RESULTS_DIR) / 'models'
if not results_dir.exists():
    print('No results found.')
else:
    print(f"{'Model':<30} {'Clean Acc':>10} {'Macro F1':>10} {'Feat Adv':>10} {'Seq PGD':>10} {'Seq FGSM':>10} {'Hybrid':>10}")
    print(f"{'-'*100}")

    for rd in sorted(results_dir.iterdir()):
        rf = rd / 'results.json'
        if not rf.exists():
            continue
        with open(rf) as f:
            res = json.load(f)

        model = rd.name
        clean_acc = res.get('test_accuracy_clean', 0)
        macro_f1 = res.get('clean_metrics', {}).get('macro_f1', 0)

        adv = res.get('adversarial_results', {})
        feat_acc = adv.get('feature_level', {}).get('accuracy', 0)
        pgd_acc = adv.get('sequence_pgd', {}).get('accuracy', 0)
        fgsm_acc = adv.get('sequence_fgsm', {}).get('accuracy', 0)
        hybrid_acc = adv.get('hybrid', {}).get('accuracy', 0)

        print(f"{model:<30} {clean_acc:>10.4f} {macro_f1:>10.4f} {feat_acc:>10.4f} {pgd_acc:>10.4f} {fgsm_acc:>10.4f} {hybrid_acc:>10.4f}")

    print(f"{'-'*100}")
    print('\n✅ Comparison complete.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION: Entraînement avec Nouvelles Contre-Mesures
# ═══════════════════════════════════════════════════════════════════════════════

# Utilisation des paramètres définis en Cell 2
if DATASETS in ['csv', 'both'] and 'csv_data' in globals() and csv_data is not None:
    print("\n" + "#"*80)
    print("  ENTRAÎNEMENT AVEC CONTRE-MESURES - LSTM sur CSV")
    print("#"*80 + "\n")
    
    results = train_with_countermeasures(
        model_type='lstm',
        data_dict=csv_data,
        lambda_trades=LAMBDA_TRADES,
        epsilon=TRADES_EPSILON,
        alpha=TRADES_EPSILON / 4,
        pgd_steps=TRADES_PGD_STEPS,
        epochs_phase1=20,
        epochs_phase2=PHASE2_EPOCHS,
        batch_size=BATCH_SIZE,
        use_input_transform=USE_INPUT_TRANSFORM,
        use_cutmix=USE_CUTMIX,
        save_dir=f'{DRIVE_RESULTS_DIR}/models/lstm_countermeasures_csv',
        use_class_weights=True,
    )
    
    print(f"\n✅ Entraînement terminé!")
    print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
    print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
    print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")
else:
    print("⚠️ CSV data not loaded. Run the data loading cells first.")

# 📊 Comparaison Finale: Méthodes d'Entraînement

| Méthode | Clean Acc | Adv Acc | Robustness Ratio | Contre-Mesures |
|---------|-----------|---------|------------------|----------------|
| **Standard** | ~95% | ~5% | ~0.05 | Aucune |
| **Adversarial (Phase 2)** | ~95% | ~10% | ~0.10 | Sensitivity + Search |
| **TRADES** | ~92% | ~40% | ~0.43 | TRADES Loss |
| **Contre-Mesures (Recommandé)** | ~90% | ~50% | ~0.55 | TRADES + Transform + CutMix |

## Améliorations Clés

1. **ADV_RATIO 0.4**: Plus d'exemples adverses pendant l'entraînement
2. **PHASE2_EPOCHS 25**: Plus de temps pour converger vers la robustesse
3. **TRADES (λ=6.0)**: Trade-off accuracy/robustesse optimal
4. **Input Transform**: Regularisation par bruit et dropout
5. **Adversarial CutMix**: Augmentation mixte clean/adversarial